# Residual stress by the sin²ψ method

A symmetric θ–2θ scan sees only the planes parallel to the surface, and a stress lying *in* the
surface reaches them only through the Poisson contraction — a strain of 10⁻⁴ that no one can tell
apart from a change of composition. This notebook measures a stress the way a diffractometer does:
by tilting the specimen so that inclined planes diffract, and reading the stress from how their
spacing changes with inclination.

It uses a **synthetic measurement of a known stress**, so that every number can be checked against
the truth, and then shows how each ingredient — the elastic constants, the stress-free spacing,
the Kα doublet, out-of-plane shear — moves the answer. The theory is in
{doc}`../../theory/residual_stress_sin2psi`; the workbench view is **XRD → Residual stress (sin²ψ)**.

Standard uncertainty is written u(x) throughout, because σ is the stress.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

from pytex.core.fixtures import get_phase_fixture
from pytex import FrameDomain, ReferenceFrame
from pytex.diffraction.xrd import RadiationSpec
from pytex.diffraction.xrd_residual_stress import (
    DiffractionElasticConstants,
    determine_residual_stress,
    locate_stress_peak,
    measurement_direction,
    parse_stress_peak_positions,
    residual_stress_pipeline,
    simulate_sin2psi_measurement,
    single_crystal_stiffness,
)

# The pinned ferrite of the fixture corpus (COD 9008536), not a hand-typed cell.
CRYSTAL = ReferenceFrame("crystal", FrameDomain.CRYSTAL, ("a", "b", "c"))
ferrite = get_phase_fixture("fe_bcc").load_phase(crystal_frame=CRYSTAL)
radiation = RadiationSpec.cr_ka()
d0 = ferrite.lattice.a / math.sqrt(6.0)  # the (211) spacing of the stress-free cell
two_theta = math.degrees(2 * math.asin(radiation.wavelength_angstrom / (2 * d0)))
print(f"d0(211) = {d0:.6f} Å, reflecting Cr Kα1 at 2θ = {two_theta:.2f}°")

## 1. The geometry: which planes are measured

At azimuth φ (in the surface, from S1 towards S2) and tilt ψ (from the surface normal S3) the planes
that diffract have their normal along

m = (cos φ sin ψ, sin φ sin ψ, cos ψ),

and their strain is m·ε·m. The high Bragg angle is deliberate: strain sensitivity grows as tan θ, which
is why Cr Kα on ferrite (211) at 156° is the classical stress measurement.

In [ ]:
for phi, psi in ((0, 0), (0, 45), (90, 45), (45, -30)):
    print(f"phi = {phi:>3} deg, psi = {psi:>4} deg:  m = {np.round(measurement_direction(phi, psi), 4)}")

## 2. Diffraction elastic constants belong to the reflection

The strain of the grains that diffract is linear in the stress through two constants of the reflection,
ε = ½S₂ m·σ·m + S₁ tr σ. For an isotropic solid ½S₂ = (1 + ν)/E and S₁ = −ν/E; for a real crystal they
depend on the reflection and on how the grains share the load. The Reuss and Voigt models bound
them; Kröner's self-consistent model lies between and is usually closest to measurement.

In [ ]:
stiffness = single_crystal_stiffness("fe_bcc")
reflections = [(2, 0, 0), (3, 1, 0), (2, 1, 1), (2, 2, 2)]
print(f"{'hkl':>6} {'Reuss':>8} {'Hill':>8} {'Kroener':>8} {'Voigt':>8}   (1/2 S2 in 1/TPa)")
for hkl in reflections:
    row = [
        DiffractionElasticConstants.for_reflection(ferrite, hkl, stiffness, model=model).half_s2_per_tpa
        for model in ("reuss", "hill", "kroener", "voigt")
    ]
    print(f"{''.join(map(str, hkl)):>6} " + " ".join(f"{value:8.3f}" for value in row))
isotropic = DiffractionElasticConstants.isotropic(210.0, 0.28)
print(f"isotropic steel (E = 210 GPa, nu = 0.28): 1/2 S2 = {isotropic.half_s2_per_tpa:.3f} /TPa")

Between (200) and (222) the Reuss constant changes by more than a factor of two — a stress
computed with the wrong reflection's constants is wrong by the same factor. For (211) all four
models nearly agree, which is one more reason (211) is the reflection of choice. We use the Kröner
constants with a 5 % relative uncertainty.

In [ ]:
dec = DiffractionElasticConstants.for_reflection(
    ferrite, (2, 1, 1), stiffness, model="kroener", relative_standard_uncertainty=0.05
)
print(dec.describe())

## 3. A measurement of a known stress

A shot-peened surface: σ11 = −350, σ22 = −150 and σ12 = 60 MPa. Three azimuths, nine tilts of both
signs spaced evenly in sin²ψ, each scan a Kα1/Kα2 pseudo-Voigt doublet broadened as 1/cos ψ
(defocusing under ω-tilting), shaped by the Lorentz–polarization–absorption factor, with Poisson
counting noise.

In [ ]:
truth = {"sigma_11": -350.0, "sigma_22": -150.0, "sigma_12": 60.0}
tilts = (-45.0, -37.8, -30.0, -21.1, 0.0, 21.1, 30.0, 37.8, 45.0)
measurement = simulate_sin2psi_measurement(
    d0_angstrom=d0, stress_mpa=truth, dec=dec, psi_deg=tilts, seed=7
)
print(measurement.describe())

fig, axes = plt.subplots(1, 3, figsize=(10, 3), sharey=True)
for ax, phi in zip(axes, (0.0, 45.0, 90.0)):
    for scan in measurement.scans:
        if scan.phi_deg != phi:
            continue
        counts = scan.pattern.intensity
        ax.plot(scan.pattern.two_theta_deg, (counts - counts.min()) / np.ptp(counts),
                color=plt.cm.coolwarm((scan.psi_deg + 45) / 90), lw=0.8)
    ax.set_title(f"phi = {phi:g} deg")
    ax.set_xlabel("2θ (°)")
axes[0].set_ylabel("normalized counts")
fig.suptitle("The peak moves with tilt (blue: psi < 0, red: psi > 0)")
fig.tight_layout()

A compressive stress contracts the planes inclined to the surface, so the peak moves to *higher*
2θ with tilt — most at φ = 0°, where the stress is largest. The width growing with |ψ| is
defocusing, not stress.

## 4. Locating one peak

Every stress below is computed from peak positions, and each position's standard uncertainty is its
weight. The default fit models the Kα2 partner at its Bragg-law position rather than stripping it.

In [ ]:
scan = measurement.scans[0]
peak = locate_stress_peak(scan, expected_two_theta_deg=two_theta, window_deg=8.0)
print(peak.describe())
print(f"d = {peak.d_spacing_angstrom(radiation.wavelength_angstrom):.6f} "
      f"+/- {peak.d_spacing_uncertainty_angstrom(radiation.wavelength_angstrom):.6f} Å  "
      "(u(d) = d cot(theta) u(theta))")

## 5. The sin²ψ lines and the stress tensor

Under plane stress d = d₀[1 + ½S₂ σφ sin²ψ + S₁(σ11 + σ22)]: **linear in sin²ψ, with slope
d₀·½S₂·σφ**. The pipeline locates every peak, fits the line at each azimuth, and fits the tensor to
all 27 positions at once.

In [ ]:
result = residual_stress_pipeline(
    measurement, d0_angstrom=d0, dec=dec, window_deg=8.0,
    d0_uncertainty_angstrom=d0 * 1e-4, monte_carlo_draws=4000, seed=1,
)

fig, ax = plt.subplots(figsize=(6, 4))
for color, line in zip(("C0", "C1", "C2"), result.regressions):
    positive = line.psi_deg >= 0
    ax.errorbar(line.sin2psi[positive], line.d_angstrom[positive],
                yerr=line.d_uncertainty_angstrom[positive], fmt="o", color=color,
                label=f"phi = {line.phi_deg:g}: sigma_phi = {line.sigma_phi_mpa:.0f} MPa")
    ax.errorbar(line.sin2psi[~positive], line.d_angstrom[~positive],
                yerr=line.d_uncertainty_angstrom[~positive], fmt="o", mfc="white", color=color)
    grid = np.linspace(0, 0.5, 20)
    ax.plot(grid, line.intercept_angstrom + line.slope_angstrom * grid, color=color)
ax.set_xlabel("sin²ψ")
ax.set_ylabel("d (Å)")
ax.legend(fontsize=8)
fig.tight_layout()

fit = result.tensor
for name, value, u in zip(fit.component_names, fit.values_mpa, fit.combined_uncertainty_mpa):
    print(f"{name}: {value:8.1f} +/- {u:5.1f} MPa   (generated with {truth[name]:.0f})")
print(f"strain-fit reduced chi-squared = {fit.reduced_chi_squared:.2f}")

Three falling lines: compressive along every azimuth. The recovered components match the
generating stress. But the ± values are much larger than the scatter of the points suggests — the
next section says why.

## 6. The uncertainty budget: where the ± comes from

In [ ]:
names = fit.component_names
print(f"{'':10}" + "".join(f"{n:>10}" for n in names))
for source, values in fit.budget_mpa.items():
    print(f"{source:>10}" + "".join(f"{v:10.1f}" for v in values))
print(f"{'combined':>10}" + "".join(f"{v:10.1f}" for v in fit.combined_uncertainty_mpa))
print(f"{'MonteCarlo':>10}" + "".join(f"{v:10.1f}" for v in fit.monte_carlo_uncertainty_mpa))

The counting statistics contribute a couple of MPa. **d₀ dominates**: a relative error of 10⁻⁴ in
d₀ is a uniform strain of 10⁻⁴, which the fit can only explain through S₁(σ11 + σ22), and
10⁻⁴/1.23 TPa⁻¹ is about 80 MPa. The Monte Carlo row, which redraws every input and refits,
agrees with the linear propagation.

The slopes are nearly immune. Watch what a deliberately wrong d₀ does:

In [ ]:
wrong = residual_stress_pipeline(measurement, d0_angstrom=d0 * 1.0003, dec=dec, window_deg=8.0)
for good, bad in zip(result.regressions, wrong.regressions):
    print(f"phi = {good.phi_deg:>4g}: sigma_phi {good.sigma_phi_mpa:7.1f} -> {bad.sigma_phi_mpa:7.1f} MPa")
for name, good, bad in zip(names, fit.values_mpa, wrong.tensor.values_mpa):
    print(f"{name}: {good:7.1f} -> {bad:7.1f} MPa")
print(f"strain-fit chi2_nu: {fit.reduced_chi_squared:.2f} -> {wrong.tensor.reduced_chi_squared:.2f}")

A 3 × 10⁻⁴ error in d₀ leaves every σφ — every slope — within a fraction of a percent, but moves
σ11 and σ22 by about 150 MPa each: the joint fit reads their sum from the absolute strain, and with
azimuths of 0°, 45° and 90° it then has to bend σ12 as well to keep the slopes. No stress tensor
reconciles the true slopes with the shifted intercepts, and the strain-fit χ²ν says so. When no
stress-free reference exists, d₀ can be *determined* from the data under the plane-stress
assumption σ33 = 0:

In [ ]:
refined = residual_stress_pipeline(
    measurement, d0_angstrom=d0 * 1.0003, dec=dec, window_deg=8.0, refine_d0=True
)
print(f"refined d0 = {refined.d0_angstrom:.6f} +/- {refined.d0_uncertainty_angstrom:.6f} Å "
      f"(true {d0:.6f})")
print(np.round(refined.tensor.values_mpa, 1), "MPa")

## 7. Out-of-plane shear splits the branches

A shear stress σ13 adds ½S₂ τφ sin 2ψ, which is *odd* in ψ: the ψ > 0 and ψ < 0 measurements
separate into two branches. A biaxial evaluation cannot represent that; the evaluation with shear
components can.

In [ ]:
sheared = simulate_sin2psi_measurement(
    d0_angstrom=d0, stress_mpa={**truth, "sigma_13": 60.0}, dec=dec, psi_deg=tilts, seed=8
)
with_shear = residual_stress_pipeline(
    sheared, d0_angstrom=d0, dec=dec, window_deg=8.0, stress_state="biaxial_shear"
)
line = with_shear.regressions[0]
s2, a1, u1, a2, u2 = line.branch_averages()
fig, (left, right) = plt.subplots(1, 2, figsize=(9, 3.2))
positive = line.psi_deg >= 0
left.plot(line.sin2psi[positive], line.d_angstrom[positive], "o", label="psi > 0")
left.plot(line.sin2psi[~positive], line.d_angstrom[~positive], "o", mfc="white", label="psi < 0")
left.set_xlabel("sin²ψ"); left.set_ylabel("d (Å)"); left.legend(); left.set_title("phi = 0")
right.errorbar(np.sin(2 * np.arcsin(np.sqrt(s2))), 1e3 * a2, yerr=1e3 * u2, fmt="o")
right.set_xlabel("sin|2ψ|"); right.set_ylabel("a2 = (d+ − d−)/2 (mÅ)")
fig.tight_layout()
print(f"tau_phi at phi = 0: {line.tau_phi_mpa:.1f} +/- {line.tau_phi_uncertainty_mpa:.1f} MPa (sigma_13 = 60)")
print(np.round(with_shear.tensor.values_mpa, 1), with_shear.tensor.component_names)

## 8. Why the Kα2 line matters at stress angles

At 2θ ≈ 156° the Cr Kα doublet is almost a degree apart — comparable to the peak width, which
grows with tilt. A parabola through the top of the *blend* is biased differently at every tilt, so
the bias does not cancel in the slope. Stripping Kα2 first (Rachinger) removes it; the profile fit
models it instead.

In [ ]:
for method, strip in (("parabola", False), ("parabola", True), ("centroid", True), ("pseudo_voigt", True)):
    attempt = residual_stress_pipeline(
        measurement, d0_angstrom=d0, dec=dec, window_deg=8.0, peak_method=method, strip_doublet=strip
    )
    label = f"{method}{' (K-alpha2 stripped)' if strip and method != 'pseudo_voigt' else ''}"
    print(f"{label:>32}: sigma_11 = {attempt.tensor.values_mpa[0]:7.1f} MPa, "
          f"chi2_nu = {attempt.tensor.reduced_chi_squared:6.2f}")

## 9. Your own data

Measured peak positions (from any software) are a table of φ ψ 2θ and, optionally, u(2θ); whole
scans are φ ψ 2θ intensity with one row per point (`parse_stress_scans`). Without a u(2θ) column
the uncertainty comes from the scatter alone.

In [ ]:
table = "\n".join(
    f"{peak.phi_deg:g} {peak.psi_deg:g} {peak.two_theta_deg:.5f}" for peak in result.peaks
)
from_table = determine_residual_stress(
    parse_stress_peak_positions(table),
    wavelength_angstrom=radiation.wavelength_angstrom,
    d0_angstrom=d0,
    dec=dec,
    reflection_label="(211)",
    phase_name="ferrite",
)
print(from_table.describe())

## What this does not do

It measures the macroscopic stress averaged over the depth the X-rays reach (a few micrometres here),
for an untextured material. A stress gradient or texture makes d curved or oscillating in sin²ψ — the
curvature test in every regression flags it — and needs the τ- or scattering-vector method, or
stress factors in place of the two elastic constants.